In [1]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
df_iot = pd.read_csv('../Dataset/raw/IoTProcessed_Data.csv')

In [4]:
# Fix column name typo
df_iot.rename(columns={'tempreature': 'temperature'}, inplace=True)

# Convert date
df_iot['date'] = pd.to_datetime(df_iot['date'])

# Sort properly (IMPORTANT)
df_iot = df_iot.sort_values('date')

In [5]:
# Rolling smoothing (VERY IMPORTANT)
df_iot['temperature'] = df_iot['temperature'].rolling(3).mean()
df_iot['humidity'] = df_iot['humidity'].rolling(3).mean()
df_iot['water_level'] = df_iot['water_level'].rolling(3).mean()

In [6]:
# Replace invalid values
df_iot[['N','P','K']] = df_iot[['N','P','K']].replace(255, None)

# Fill intelligently
df_iot[['N','P','K']] = df_iot[['N','P','K']].fillna(method='ffill')

C:\Users\rajor\AppData\Local\Temp\ipykernel_21688\1693538704.py:5: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_iot[['N','P','K']] = df_iot[['N','P','K']].fillna(method='ffill')
C:\Users\rajor\AppData\Local\Temp\ipykernel_21688\1693538704.py:5: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_iot[['N','P','K']] = df_iot[['N','P','K']].fillna(method='ffill')


In [7]:
df_iot['temp_humidity_index'] = df_iot['temperature'] * df_iot['humidity']
df_iot['water_stress'] = df_iot['temperature'] / (df_iot['water_level'] + 1e-5)

In [8]:
df_iot['npk_sum'] = df_iot['N'] + df_iot['P'] + df_iot['K']
df_iot['np_ratio'] = df_iot['N'] / (df_iot['P'] + 1e-5)

Fan_actuator_ON = 1 → system cooling
Water_pump_ON = 1 → irrigation happening

In [9]:
df_iot['fan_active'] = df_iot['Fan_actuator_ON']
df_iot['irrigation_active'] = df_iot['Water_pump_actuator_ON']

In [10]:
# Count how often irrigation happens
df_iot['irrigation_events'] = df_iot['irrigation_active'].rolling(10).sum()

# Fan usage intensity
df_iot['fan_usage'] = df_iot['fan_active'].rolling(10).sum()

In [11]:
df_iot.set_index('date', inplace=True)

iot_daily = df_iot.resample('D').agg({
    'temperature': 'mean',
    'humidity': 'mean',
    'water_level': 'mean',
    'N': 'mean',
    'P': 'mean',
    'K': 'mean',
    
    # Engineered
    'temp_humidity_index': 'mean',
    'water_stress': 'mean',
    'npk_sum': 'mean',
    
    # Actuator behavior
    'fan_active': 'sum',
    'irrigation_active': 'sum',
    'irrigation_events': 'sum'
}).reset_index()

In [12]:
iot_daily['temp_change'] = iot_daily['temperature'].diff()
iot_daily['humidity_change'] = iot_daily['humidity'].diff()

iot_daily['temp_3d_avg'] = iot_daily['temperature'].rolling(3).mean()
iot_daily['water_7d_avg'] = iot_daily['water_level'].rolling(7).mean()

In [13]:
iot_daily = iot_daily.fillna(method='ffill')
iot_daily = iot_daily.fillna(method='bfill')

C:\Users\rajor\AppData\Local\Temp\ipykernel_21688\842480675.py:1: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  iot_daily = iot_daily.fillna(method='ffill')
C:\Users\rajor\AppData\Local\Temp\ipykernel_21688\842480675.py:2: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  iot_daily = iot_daily.fillna(method='bfill')


In [18]:
import os

save_path = "../Dataset/processed/iot_daily.csv"

# Create folder if not exists
os.makedirs(os.path.dirname(save_path), exist_ok=True)

In [19]:
iot_daily.to_csv(save_path, index=False)

# Agro Environmental dataset 

In [21]:

df_agro = pd.read_csv("../Dataset/raw/agro_environmental_dataset.csv")

# Drop ID (no learning value)
df_agro.drop('location_id', axis=1, inplace=True)

# Remove duplicates
df_agro = df_agro.drop_duplicates()

df_agro = df_agro.reset_index(drop=True)

In [22]:
from sklearn.preprocessing import LabelEncoder

cat_cols = [
    'soil_type',
    'moisture_regime',
    'thermal_regime',
    'nutrient_balance',
    'plant_category'
]

encoders = {}

for col in cat_cols:
    le = LabelEncoder()
    df_agro[col] = le.fit_transform(df_agro[col])
    encoders[col] = le

Soil Fertility

In [23]:
df_agro['fertility_index'] = (
    df_agro['organic_matter_pct'] +
    df_agro['cation_exchange_capacity'] +
    df_agro['nitrogen_ppm']
)

df_agro['nutrient_total'] = (
    df_agro['nitrogen_ppm'] +
    df_agro['phosphorus_ppm'] +
    df_agro['potassium_ppm']
)

Nutrient intelligence

In [24]:
df_agro['np_ratio'] = df_agro['nitrogen_ppm'] / (df_agro['phosphorus_ppm'] + 1e-5)
df_agro['nk_ratio'] = df_agro['nitrogen_ppm'] / (df_agro['potassium_ppm'] + 1e-5)
df_agro['pk_ratio'] = df_agro['phosphorus_ppm'] / (df_agro['potassium_ppm'] + 1e-5)

In [25]:
df_agro['moisture_range'] = (
    df_agro['moisture_limit_wet'] - df_agro['moisture_limit_dry']
)

df_agro['moisture_stress'] = (
    df_agro['soil_moisture_pct'] / (df_agro['moisture_limit_wet'] + 1e-5)
)

In [26]:
df_agro['temp_diff'] = df_agro['air_temp_c'] - df_agro['soil_temp_c']

df_agro['thermal_stress_index'] = (
    df_agro['air_temp_c'] / (df_agro['soil_temp_c'] + 1e-5)
)

In [27]:
df_agro['ph_deviation'] = abs(df_agro['soil_ph'] - 7)

df_agro['salinity_stress'] = (
    df_agro['salinity_ec'] * df_agro['soil_moisture_pct']
)

In [28]:
df_agro['total_stress_index'] = (
    df_agro['salinity_ec'] +
    df_agro['ph_deviation'] +
    df_agro['thermal_stress_index']
)

In [29]:
for col in df_agro.select_dtypes(include=['float64','int64']).columns:
    q1 = df_agro[col].quantile(0.25)
    q3 = df_agro[col].quantile(0.75)
    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    df_agro[col] = df_agro[col].clip(lower, upper)

In [30]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

num_cols = df_agro.drop(
    ['failure_flag', 'stress_level', 'suitability_score'],
    axis=1
).columns

df_agro[num_cols] = scaler.fit_transform(df_agro[num_cols])

In [31]:
X = df_agro.drop(['failure_flag'], axis=1)
y = df_agro['failure_flag']

In [32]:

save_path = "../Dataset/processed/agro_processed.csv"
os.makedirs(os.path.dirname(save_path), exist_ok=True)

df_agro.to_csv(save_path, index=False)

# Smart Farming Crop Yield

In [33]:
df_smart = pd.read_csv("../Dataset/raw/Smart_Farming_Crop_Yield_2024.csv")

# Convert dates
df_smart['sowing_date'] = pd.to_datetime(df_smart['sowing_date'])
df_smart['harvest_date'] = pd.to_datetime(df_smart['harvest_date'])
df_smart['timestamp'] = pd.to_datetime(df_smart['timestamp'])

# Fix column names (clean)
df_smart.rename(columns={
    'soil_moisture_%': 'soil_moisture',
    'humidity_%': 'humidity'
}, inplace=True)

# Drop duplicates
df_smart = df_smart.drop_duplicates().reset_index(drop=True)

In [34]:
df_smart['irrigation_type'].fillna('Unknown', inplace=True)
df_smart['crop_disease_status'].fillna('None', inplace=True)

C:\Users\rajor\AppData\Local\Temp\ipykernel_21688\2079694524.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_smart['irrigation_type'].fillna('Unknown', inplace=True)
C:\Users\rajor\AppData\Local\Temp\ipykernel_21688\2079694524.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a cop

In [35]:
# Recalculate duration (trust your calculation more)
df_smart['growth_days_calc'] = (
    df_smart['harvest_date'] - df_smart['sowing_date']
).dt.days

# Efficiency
df_smart['yield_per_day'] = (
    df_smart['yield_kg_per_hectare'] / (df_smart['growth_days_calc'] + 1)
)

In [36]:
df_smart['climate_score'] = (
    df_smart['temperature_C'] +
    df_smart['humidity'] +
    df_smart['rainfall_mm']
)

df_smart['water_efficiency'] = (
    df_smart['rainfall_mm'] / (df_smart['soil_moisture'] + 1e-5)
)

vegetation intelligence

In [37]:
df_smart['ndvi_health_index'] = df_smart['NDVI_index'] * df_smart['soil_moisture']

In [38]:
df_smart['input_intensity'] = (
    df_smart['pesticide_usage_ml']
)

In [39]:
df_smart['heat_stress'] = df_smart['temperature_C'] - 25

df_smart['ph_deviation'] = abs(df_smart['soil_pH'] - 7)

In [40]:


cat_cols = [
    'region',
    'crop_type',
    'irrigation_type',
    'fertilizer_type',
    'crop_disease_status'
]

encoders = {}

for col in cat_cols:
    le = LabelEncoder()
    df_smart[col] = le.fit_transform(df_smart[col])
    encoders[col] = le

Time anchors

In [41]:
df_smart['mid_date'] = df_smart['sowing_date'] + (
    (df_smart['harvest_date'] - df_smart['sowing_date']) / 2
)

In [42]:
y = df_smart['yield_kg_per_hectare']
X = df_smart.drop(['yield_kg_per_hectare'], axis=1)

In [43]:


num_cols = df_smart.select_dtypes(include=['float64','int64']).columns

scaler = StandardScaler()
df_smart[num_cols] = scaler.fit_transform(df_smart[num_cols])

In [44]:

save_path = "../Dataset/processed/smart_farming_processed.csv"
os.makedirs(os.path.dirname(save_path), exist_ok=True)

df_smart.to_csv(save_path, index=False)

NASA Datasets

1) Bangalore
2) Chennai
3) Delhi
4) Kolkata

In [52]:

import glob

folder_path = "..\\Dataset\\raw\\NASA\\"
files = glob.glob(os.path.join(folder_path, "*_NASA.csv"))
print(files)


['..\\Dataset\\raw\\NASA\\Bangalore_NASA.csv', '..\\Dataset\\raw\\NASA\\chennai_NASA.csv', '..\\Dataset\\raw\\NASA\\Delhi_NASA.csv', '..\\Dataset\\raw\\NASA\\Kolkata_NASA.csv']


In [53]:
def load_nasa_file(file):
    
    # Step 1: find header row (skip metadata)
    with open(file, 'r') as f:
        lines = f.readlines()
    
    for i, line in enumerate(lines):
        if line.startswith("YEAR"):
            header_row = i
            break
    
    # Step 2: read actual data
    df = pd.read_csv(file, skiprows=header_row)
    
    # Step 3: replace -999 with NaN
    num_cols = df.select_dtypes(include=['float64','int64']).columns
    df[num_cols] = df[num_cols].replace(-999, np.nan)
    
    return df

In [54]:
dfs = []

for file in files:
    df = load_nasa_file(file)
    
    # Extract city name
    city = os.path.basename(file).split("_")[0].lower()
    
    df['location'] = city
    
    dfs.append(df)

# Combine all cities
nasa_df = pd.concat(dfs).reset_index(drop=True)

print(nasa_df.head())

   YEAR  DOY   RH2M  ALLSKY_SFC_SW_DWN    T2M   location
0  2020    1  74.76              11.64  23.05  bangalore
1  2020    2  73.83              14.50  23.29  bangalore
2  2020    3  71.56              17.82  24.06  bangalore
3  2020    4  71.99              17.08  24.46  bangalore
4  2020    5  73.33              16.80  24.23  bangalore


In [55]:
nasa_df['date'] = pd.to_datetime(nasa_df['YEAR'], format='%Y') + \
                  pd.to_timedelta(nasa_df['DOY'] - 1, unit='D')

nasa_df = nasa_df.sort_values(['location', 'date'])

In [56]:
nasa_df.rename(columns={
    'RH2M': 'humidity',
    'ALLSKY_SFC_SW_DWN': 'solar_radiation',
    'T2M': 'temperature'
}, inplace=True)

In [57]:
nasa_df = nasa_df[
    ['date', 'location', 'temperature', 'humidity', 'solar_radiation']
]

In [58]:
# Forward fill per location
nasa_df = nasa_df.groupby('location').apply(
    lambda x: x.fillna(method='ffill').fillna(method='bfill')
).reset_index(drop=True)

print(nasa_df.isna().sum())

date               0
location           0
temperature        0
humidity           0
solar_radiation    0
dtype: int64


C:\Users\rajor\AppData\Local\Temp\ipykernel_21688\2410341554.py:3: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  lambda x: x.fillna(method='ffill').fillna(method='bfill')
C:\Users\rajor\AppData\Local\Temp\ipykernel_21688\2410341554.py:2: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  nasa_df = nasa_df.groupby('location').apply(


In [59]:
nasa_df['temp_change'] = nasa_df.groupby('location')['temperature'].diff()

nasa_df['temp_3d_avg'] = nasa_df.groupby('location')['temperature'].transform(
    lambda x: x.rolling(3).mean()
)

In [60]:
nasa_df['solar_temp_interaction'] = (
    nasa_df['solar_radiation'] * nasa_df['temperature']
)

In [61]:
nasa_df['humidity_stress'] = (
    nasa_df['humidity'] / (nasa_df['temperature'] + 1e-5)
)


In [62]:
nasa_df['temp_lag1'] = nasa_df.groupby('location')['temperature'].shift(1)

nasa_df['humidity_lag2'] = nasa_df.groupby('location')['humidity'].shift(2)

In [63]:
nasa_df = nasa_df.groupby('location').apply(
    lambda x: x.fillna(method='ffill').fillna(method='bfill')
).reset_index(drop=True)

C:\Users\rajor\AppData\Local\Temp\ipykernel_21688\3725047843.py:2: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  lambda x: x.fillna(method='ffill').fillna(method='bfill')
C:\Users\rajor\AppData\Local\Temp\ipykernel_21688\3725047843.py:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  nasa_df = nasa_df.groupby('location').apply(


In [64]:
print("Missing values:\n", nasa_df.isna().sum())
print("\nShape:", nasa_df.shape)
print(nasa_df.head())

Missing values:
 date                      0
location                  0
temperature               0
humidity                  0
solar_radiation           0
temp_change               0
temp_3d_avg               0
solar_temp_interaction    0
humidity_stress           0
temp_lag1                 0
humidity_lag2             0
dtype: int64

Shape: (9132, 11)
        date   location  temperature  humidity  solar_radiation  temp_change  \
0 2020-01-01  bangalore        23.05     74.76            11.64         0.24   
1 2020-01-02  bangalore        23.29     73.83            14.50         0.24   
2 2020-01-03  bangalore        24.06     71.56            17.82         0.77   
3 2020-01-04  bangalore        24.46     71.99            17.08         0.40   
4 2020-01-05  bangalore        24.23     73.33            16.80        -0.23   

   temp_3d_avg  solar_temp_interaction  humidity_stress  temp_lag1  \
0    23.466667                268.3020         3.243383      23.05   
1    23.466667        

In [65]:
save_path = "../Dataset/processed/nasa_processed.csv"
os.makedirs(os.path.dirname(save_path), exist_ok=True)

nasa_df.to_csv(save_path, index=False)

print("Saved at:", save_path)

Saved at: ../Dataset/processed/nasa_processed.csv
